# eBay Tech Deals – Exploratory Data Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('cleaned_ebay_deals.csv')
print(df.shape)
df.head()

## 1. Time Series Analysis

In [ ]:
# Convert timestamp and sort
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp')

# Extract hour and count deals per hour
df['hour'] = df['timestamp'].dt.hour
deals_per_hour = df.groupby('hour').size().reset_index(name='count')

plt.figure(figsize=(12, 5))
plt.bar(deals_per_hour['hour'], deals_per_hour['count'], color='steelblue', edgecolor='black')
plt.title('Number of Deals Scraped Per Hour')
plt.xlabel('Hour of Day (0–23)')
plt.ylabel('Number of Deals')
plt.xticks(range(0, 24))
plt.tight_layout()
plt.savefig('deals_per_hour.png', dpi=150)
plt.show()

## 2. Price and Discount Analysis

In [ ]:
# Histogram of prices
plt.figure(figsize=(10, 4))
plt.hist(df['price'].dropna(), bins=40, color='coral', edgecolor='black')
plt.title('Distribution of Product Prices')
plt.xlabel('Price (USD)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('price_histogram.png', dpi=150)
plt.show()

In [ ]:
# Boxplot of prices
plt.figure(figsize=(8, 4))
plt.boxplot(df['price'].dropna(), vert=False, patch_artist=True,
            boxprops=dict(facecolor='lightblue', color='navy'))
plt.title('Boxplot of Product Prices')
plt.xlabel('Price (USD)')
plt.tight_layout()
plt.savefig('price_boxplot.png', dpi=150)
plt.show()

In [ ]:
# Scatter plot: original_price vs price
plt.figure(figsize=(8, 6))
plt.scatter(df['original_price'], df['price'], alpha=0.4, color='purple', edgecolors='none')
max_val = max(df['original_price'].max(), df['price'].max())
plt.plot([0, max_val], [0, max_val], 'r--', label='price = original_price')
plt.title('Original Price vs Discounted Price')
plt.xlabel('Original Price (USD)')
plt.ylabel('Discounted Price (USD)')
plt.legend()
plt.tight_layout()
plt.savefig('price_scatter.png', dpi=150)
plt.show()

In [ ]:
# Distribution of discount_percentage
plt.figure(figsize=(10, 4))
plt.hist(df['discount_percentage'].dropna(), bins=40, color='green', edgecolor='black')
plt.title('Distribution of Discount Percentages')
plt.xlabel('Discount (%)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('discount_distribution.png', dpi=150)
plt.show()

## 3. Shipping Information Analysis

In [ ]:
shipping_counts = df['shipping'].value_counts().head(10)

plt.figure(figsize=(12, 5))
shipping_counts.plot(kind='bar', color='teal', edgecolor='black')
plt.title('Top 10 Shipping Options by Frequency')
plt.xlabel('Shipping Option')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('shipping_frequency.png', dpi=150)
plt.show()

## 4. Text Analysis on Product Titles

In [ ]:
keywords = ['Apple', 'Samsung', 'Laptop', 'iPhone', 'Tablet', 'Gimbal']

keyword_counts = {
    kw: df['title'].dropna().str.contains(kw, case=False, regex=False).sum()
    for kw in keywords
}

kw_series = pd.Series(keyword_counts).sort_values(ascending=False)
print(kw_series)

plt.figure(figsize=(8, 5))
colors = sns.color_palette('pastel', len(kw_series))
plt.bar(kw_series.index, kw_series.values, color=colors, edgecolor='black')
plt.title('Keyword Frequency in Product Titles')
plt.xlabel('Keyword')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('keyword_frequency.png', dpi=150)
plt.show()

## 5. Price Difference Analysis

In [ ]:
df['absolute_discount'] = df['original_price'] - df['price']

plt.figure(figsize=(10, 4))
plt.hist(df['absolute_discount'].dropna(), bins=40, color='orange', edgecolor='black')
plt.title('Distribution of Absolute Price Differences (Original - Discounted)')
plt.xlabel('Price Difference (USD)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('price_difference.png', dpi=150)
plt.show()

## 6. Top 5 Deals by Discount Percentage

In [ ]:
top5 = (
    df[['title', 'price', 'original_price', 'discount_percentage']]
    .dropna(subset=['discount_percentage'])
    .sort_values('discount_percentage', ascending=False)
    .head(5)
    .reset_index(drop=True)
)

top5.index += 1  # start ranking from 1
print(top5.to_string())
top5